# Cost Controls on the AI Gateway: a per-project budget that blocks runaway spend

An agent stuck in a retry loop, or a prompt that drags a huge context into every turn, can spend a
month of model budget in an afternoon. **Cost Controls** give each Highflame project a daily and a
monthly budget in US dollars. When an agent's model calls go through the Highflame AI Gateway,
the gateway reports every call, Highflame prices it and adds it to the project's spend, and the
gateway blocks new prompts once the budget is used up. **Nothing in the agent changes.**

This notebook spends real money on a tiny budget and watches it stop:

1. Read the project's budget, without spending anything.
2. Make a model call through the gateway with the ordinary OpenAI client.
3. Keep going until the budget is used up, and see the next prompt blocked.
4. See where the money went, by agent and by model.

Agents that call their model directly and use the Highflame SDK have their own notebook:
[`02-custom-agents`](../02-custom-agents/).

## How it works

1. The agent sends its model call to the gateway, which forwards it to the provider.
2. When the provider answers, the gateway reports the call's token counts to Highflame.
3. Highflame prices the call from one price table and adds it to the project's spend for the day
   and for the month (UTC).
4. On the **next** prompt, the gateway asks Highflame for a decision. The Cost Controls policy
   compares the spend with the budget and blocks once it is used up.

The price of a call is known only after the model answers, so the call that crosses the budget
completes and the one after it is blocked. If that overshoot matters, use the **Monthly Budget
Headroom** template, which blocks at 95% of the budget.

## Before you start: set it up once in Studio

Use a project that nothing else depends on, and only this notebook. When the budget is used up,
**every** prompt in the project is blocked until the next UTC day, or until you raise the budget.

1. Sign in at [studio.highflame.ai](https://studio.highflame.ai) and create a project, for example
   `cost-controls-ai-gateway`.
2. **AI Gateway → Policies → Cost Controls → Set budget.** Set **Daily budget (USD)** to `0.05`, leave
   the monthly budget blank, and select **Deploy**. The budget belongs to the project, so the Custom
   Agents and AI Gateway pages show the same one.
3. In the same tab, deploy the **Daily Budget** template. The deploy dialog starts on **Monitor**,
   which only records; set **Enforcement Mode** to **Enforce**.
4. In the same project, create an API key: **AI Gateway → Settings → API Keys**. The notebook uses it to register an agent and to read the budget.

With `gpt-4o` each call below costs about one cent, so a $0.05 budget runs out in a handful of
calls. That is the point: the notebook should hit the limit in front of you.

## Setup

Copy `.env.example` to `.env` beside this notebook.

| Variable | What it is |
| --- | --- |
| `HIGHFLAME_API_KEY` | **Required.** The API key from step 4, in the project from step 1. |
| `PROVIDER_API_KEY` | **Required.** Your OpenAI API key. The gateway forwards it upstream. `OPENAI_API_KEY` works as a fallback. |
| `HIGHFLAME_GATEWAY_BASE_URL` | The LLM Base URL from **AI Gateway → LLM Providers**, with `/v1` appended. Defaults to `https://gateway.highflame.ai/llm/v1`. |
| `MODEL` | An OpenAI model. Defaults to `gpt-4o`, which costs enough per call to reach a small budget quickly. |
| `DAILY_BUDGET_USD` | Optional. The notebook reads the budget amount from Highflame; this is used only if Highflame does not report it. Defaults to `0.05`. |

In [1]:
%pip install -q -r requirements.txt

from importlib.metadata import version

print("highflame", version("highflame"))

Note: you may need to restart the kernel to use updated packages.
highflame 0.10.0


In [2]:
import json
import os
import time
import urllib.parse
import urllib.request
import uuid
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv
from highflame import Highflame
from openai import OpenAI

load_dotenv()

HIGHFLAME_API_KEY = os.environ["HIGHFLAME_API_KEY"]
PROVIDER_API_KEY = os.environ.get("PROVIDER_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
if not PROVIDER_API_KEY:
    raise RuntimeError("Set PROVIDER_API_KEY (or OPENAI_API_KEY) in .env to your OpenAI API key.")
GATEWAY_BASE_URL = os.environ.get(
    "HIGHFLAME_GATEWAY_BASE_URL", "https://gateway.highflame.ai/llm/v1"
).rstrip("/")
MODEL = os.environ.get("MODEL", "gpt-4o")
DAILY_BUDGET_USD = float(os.environ.get("DAILY_BUDGET_USD", "0.05"))

# The platform API and the identity service. Leave both unset for the hosted product.
API_URL = os.environ.get("HIGHFLAME_API_URL", "https://api.highflame.ai").rstrip("/")
AUTH_URL = os.environ.get("HIGHFLAME_AUTH_URL", "https://auth.highflame.ai").rstrip("/")
SDK_URLS = {"base_url": API_URL, "identity_base_url": AUTH_URL, "token_url": f"{AUTH_URL}/oauth2/token"}

RUN_ID = uuid.uuid4().hex[:6]
CREATED: list[tuple[str, str]] = []  # (label, identity id) for the clean-up cell

highflame =Highflame(api_key=HIGHFLAME_API_KEY, **SDK_URLS)
print("account:", highflame.whoami()["account_id"])
print("model  :", MODEL)

account: 757038846364
model  : gpt-4o


### An agent with its own key

Spend is counted per project, and also per agent when the caller is a registered agent. Register one,
so the report at the end can say which agent spent the money.

In [3]:
gateway_agent = highflame.agents.register(
    name="Gateway Agent",
    external_id=f"cost-gateway-{RUN_ID}",
    identity_type="agent",
    sub_type="tool_agent",
    trust_level="first_party",
    framework="openai",
    description="Cost Controls notebook demo. Safe to delete.",
)
CREATED.append(("Gateway Agent", gateway_agent.agent.id))

print("registered:", gateway_agent.agent.name)

registered: Gateway Agent


### Today's spend so far

Highflame keeps one record per priced call and serves them as a chargeback report. Read today's
totals now, before this run spends anything, so section 4 can show what this run added.

In [4]:
form = urllib.parse.urlencode({"grant_type": "api_key", "api_key": HIGHFLAME_API_KEY}).encode()
with urllib.request.urlopen(
    urllib.request.Request(f"{AUTH_URL}/oauth2/token", data=form, method="POST",
                           headers={"Content-Type": "application/x-www-form-urlencoded"}),
    timeout=30,
) as response:
    read_token = json.loads(response.read())["access_token"]


def chargeback(group_by: str) -> dict:
    now = datetime.now(timezone.utc)
    query = urllib.parse.urlencode({
        "start": now.strftime("%Y-%m-%dT00:00:00Z"),
        "end": (now + timedelta(minutes=1)).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "group_by": group_by,
    })
    request = urllib.request.Request(
        f"{API_URL}/v1/obs/costs/chargeback?{query}", headers={"Authorization": f"Bearer {read_token}"}
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.loads(response.read())


BASELINE = chargeback("agent")
CALLS_MADE = 0  # priced calls this run makes; section 4 waits until the report has all of them
print(f"Today so far: ${BASELINE.get('total_cost_usd', 0):.4f} over {BASELINE.get('total_calls', 0)} calls")

Today so far: $0.0000 over 0 calls


## 1. Read the budget, without spending anything

`budget_state()` asks Highflame how much of the project's budget is used, the same way a policy
sees it. It runs a guard check in `monitor` mode with `debug=True`, which returns the values the
Cost Controls policies read:

| Value | Meaning |
| --- | --- |
| `budget_cost_pct_daily` | Share of today's budget used, 0 to 100 |
| `budget_exceeded_daily` | `True` once today's budget is used up |
| `agent_spend_cents_daily` | What the calling agent spent today, in US cents |

They appear only when they apply: the budget values only when the project has a daily budget, and
the agent value only when the caller is a registered agent.

The budget amount itself comes from Highflame too, through a `dryrun` guard check that returns the
configuration of every detector the active policies need. Edit the budget in Studio and the next
read shows the new amount.

The gateway itself needs no Highflame library. This notebook uses the SDK only to read the budget.

In [5]:
from highflame import GuardRequest

BUDGET_KEYS = (
    "budget_cost_pct_daily",
    "budget_exceeded_daily",
    "budget_cost_pct_monthly",
    "budget_exceeded_monthly",
    "agent_spend_cents_daily",
    "agent_spend_cents_monthly",
)


def budget_limits(client: Highflame = highflame) -> dict:
    # A dry run executes nothing: it returns the detectors the active policies need, with this
    # project's configuration for each. The budget checker's configuration is the budget.
    plan = client.guard.evaluate(GuardRequest(
        content="budget check", content_type="prompt", action="process_prompt", optimize=True, dryrun=True,
    )).optimization
    for detector in (plan.model_dump() if plan else {}).get("required_detectors") or []:
        if detector.get("name") == "budget_checker":
            return detector.get("config") or {}
    return {}


def budget_state(client: Highflame = highflame) -> dict:
    # mode="monitor": a read, never a block. The check costs nothing; it carries no tokens.
    decision = client.guard.evaluate_prompt("budget check", mode="monitor", debug=True)
    seen = {**(decision.context or {}), **(decision.projected_context or {})}
    state = {key: seen[key] for key in BUDGET_KEYS if key in seen}
    state["daily_usd"] = budget_limits(client).get("daily_usd", DAILY_BUDGET_USD)
    return state


def meter(state: dict) -> str:
    pct = int(state.get("budget_cost_pct_daily", 0))
    bar = "#" * (pct // 5) + "." * (20 - pct // 5)
    budget = state.get("daily_usd") or DAILY_BUDGET_USD
    used = budget * pct / 100
    flag = "  USED UP" if state.get("budget_exceeded_daily") else ""
    return f"[{bar}] {pct:3d}% of ${budget:.2f} (~${used:.3f}){flag}"


state = budget_state()
if "budget_cost_pct_daily" not in state:
    raise RuntimeError(
        "This project has no daily budget. Do step 2 of 'Before you start' in Studio, then run this cell again."
    )
print("project today:", meter(state))
if state.get("budget_exceeded_daily"):
    print(
        "\nToday's budget is already used up, probably by an earlier run. Raise the budget in Studio"
        " (Cost Controls -> Edit budget), or run this again after 00:00 UTC."
    )

project today: [....................]   0% of $0.05 (~$0.000)


## 2. A model call through the AI Gateway

The gateway speaks the OpenAI API, so the ordinary OpenAI client works. Your OpenAI key goes in
`Authorization` and is forwarded upstream; the agent's Highflame key goes in `X-Highflame-APIKey`.
There is no usage code at all: the gateway reports every call it forwards.

When the gateway blocks a prompt it still answers with a normal completion, so an existing client
keeps working. The completion id then starts with `chatcmpl-blocked`.

Every prompt below starts with a unique line. That keeps the provider's prompt cache out of the
way, so each call costs about the same.

In [6]:
# A support backlog to summarise: about 3,000 tokens of input, so each gpt-4o call costs about a cent.
ISSUES = ["login fails", "invoice is wrong", "export times out", "app crashes on start", "password reset email never arrives"]
PRODUCTS = ["the mobile app", "the web portal", "the billing page", "the admin console"]
BACKLOG = "\n".join(
    f"Ticket {n}: a customer reports that {ISSUES[n % 5]} in {PRODUCTS[n % 4]} since the last release."
    for n in range(1, 151)
)
TASK = "Write a short paragraph on each of the five most common problems in these tickets, with how many tickets mention it."


def work_prompt(step: int) -> str:
    return f"Request {step} of run {RUN_ID}.\n\n{TASK}\n\n{BACKLOG}"


gateway = OpenAI(
    base_url=GATEWAY_BASE_URL,
    api_key=PROVIDER_API_KEY,  # -> Authorization, forwarded upstream to OpenAI
    default_headers={"X-Highflame-APIKey": gateway_agent.api_key},
)


def gateway_call(step: int) -> dict:
    global CALLS_MADE
    reply = gateway.chat.completions.create(
        model=f"openai/{MODEL}",  # the gateway routes on provider/model
        messages=[{"role": "user", "content": work_prompt(step)}],
        max_tokens=400,
    )
    allowed = not reply.id.startswith("chatcmpl-blocked")
    CALLS_MADE += allowed  # the gateway reports each call it forwarded
    usage = reply.usage
    return {
        "allowed": allowed,
        "tokens": f"{usage.prompt_tokens}/{usage.completion_tokens}" if (allowed and usage) else "-",
        "cost": "reported by the gateway" if allowed else "-",
        "text": reply.choices[0].message.content or "",
    }


result = gateway_call(1)
print("allowed:", result["allowed"], " tokens in/out:", result["tokens"])
print("reply  :", result["text"][:200].replace("\n", " "), "...")
time.sleep(2)  # the gateway reports usage just after it answers
print("\nproject today:", meter(budget_state()))

allowed: True  tokens in/out: 3101/358
reply  : The issue with invoices being incorrect is one of the most frequently reported problems, mentioned in 36 tickets. This issue appears across various platforms, including the web portal, billing page, a ...



project today: [####................]  22% of $0.05 (~$0.011)


## 3. Keep going until the budget is used up

Each call adds to the project's spend. Once the bar reaches 100%, the next prompt is blocked.

In [7]:
refusal = None
previous = budget_state()
for step in range(2, 31):  # stops at the first refusal; the cap only bounds a misconfigured run
    result = gateway_call(step)
    time.sleep(2)  # let the usage report land before the next read
    state = budget_state()
    print(
        f"{step:>2}  {'allowed' if result['allowed'] else 'BLOCKED':<8}"
        f"  tokens {result['tokens']:<9}  {result['cost']:<24}  {meter(state)}"
    )
    if not result["allowed"]:
        refusal = result["text"]
        break
    if previous.get("budget_exceeded_daily"):
        print(
            "\nThe budget was already used up, but this call was allowed. Check that the Daily Budget"
            " template is deployed in Enforce mode, not Monitor."
        )
        break
    previous = state

print("\nWhat the agent got back:\n ", (refusal or "(not blocked)")[:300])

 2  allowed   tokens 3101/375   reported by the gateway   [#########...........]  45% of $0.05 (~$0.022)


 3  allowed   tokens 3101/344   reported by the gateway   [#############.......]  68% of $0.05 (~$0.034)


 4  allowed   tokens 3101/396   reported by the gateway   [##################..]  91% of $0.05 (~$0.045)


 5  allowed   tokens 3101/340   reported by the gateway   [####################] 100% of $0.05 (~$0.050)  USED UP


 6  BLOCKED   tokens -          -                         [####################] 100% of $0.05 (~$0.050)  USED UP

What the agent got back:
  Highflame Security: Prompt blocked: this project has used its daily AI budget.


The gateway refused the prompt before it reached OpenAI, so the refused call cost nothing. The
refusal is a normal completion, so the calling code did not break.

## 4. Where the money went

Highflame keeps one record per priced call, and serves it as a chargeback report grouped by agent,
by user, or by model. Studio shows the same report under **Observatory → Chargeback**.

The report is built from a stream, so the newest calls can take a minute or two to appear. The cell
waits until every call this run made is in it. The report covers whole UTC days, so the rows below
are today's totals; the first line is this run alone.

In [8]:
for _ in range(24):  # up to about 4 minutes
    by_agent = chargeback("agent")
    if by_agent.get("total_calls", 0) >= BASELINE.get("total_calls", 0) + CALLS_MADE:
        break
    time.sleep(10)


def show(title: str, report: dict) -> None:
    print(title)
    for row in report.get("rows", []):
        if row["cost_usd"] > 0:  # calls at $0 (an unpriced model, for one) carry no cost to explain
            label = row["key"] or "(not a registered agent)"
            print(f"  {label:<28} ${row['cost_usd']:.4f}  {row['calls']:>3} calls  {row['share']:.0%}")


run_cost = by_agent.get("total_cost_usd", 0) - BASELINE.get("total_cost_usd", 0)
print(f"This run: ${run_cost:.4f} over {CALLS_MADE} calls")
print(f"Today, this project: ${by_agent.get('total_cost_usd', 0):.4f} over {by_agent.get('total_calls', 0)} calls\n")
show("By agent (today)", by_agent)
show("\nBy model (today)", chargeback("model"))

This run: $0.0569 over 5 calls
Today, this project: $0.0569 over 5 calls

By agent (today)
  cost-gateway-486a0b          $0.0569    5 calls  100%



By model (today)
  openai/gpt-4o                $0.0569    5 calls  100%


## 5. Per-agent spend

Highflame also counts each registered agent's spend. To cap one agent, attach the **Agent Monthly
Budget** template to it under **Custom Agents**; its spend still counts against the project's
budget too.

In [9]:
gateway_hf = Highflame(api_key=gateway_agent.api_key, **SDK_URLS)
cents = budget_state(gateway_hf).get("agent_spend_cents_daily")
print(f"Gateway Agent spent today: {cents if cents is not None else '-'} cents")

Gateway Agent spent today: 5 cents


## Clean up, and run it again

The agent is deactivated rather than erased, so its name stays taken; that is why the name carries
the run id.

The project stays blocked for the rest of the UTC day. To run the notebook again today, raise the
budget in Studio (**Cost Controls → Edit budget**). To turn budgets off, remove the budget: with no
budget, the Cost Controls templates never fire.

In [10]:
for label, identity_id in reversed(CREATED):
    try:
        highflame.agents.delete(identity_id)
        print("deleted:", label)
    except Exception as exc:
        print(f"clean-up skipped for {label}: {str(exc)[:60]}")

deleted: Gateway Agent


## What this showed

- A daily budget in US dollars stopped an agent's model calls through the AI Gateway once it was used
  up, with no change to the agent and no Highflame library in its path.
- Highflame priced every call from one price table, and the gateway refused the next prompt before it
  reached the provider.
- The chargeback report named the agent and the model behind every cent.

**Good to know**

- Spend is known only after a model answers, so the call that crosses the budget completes. The
  Monthly Budget Headroom template blocks at 95% to keep the overshoot small.
- A model that is not in Highflame's price table is counted at $0.
- Days and months are UTC.
- ChatGPT and Claude subscription traffic is paid for by the subscription, so it is not counted.